In [ ]:
import os


project_dir = '/blue/shenhaowang/qingqisong/be-and-active-travel'
os.chdir(project_dir)

In [ ]:
"""
FIX: Patch torch._pytree compatibility issue
Must run this BEFORE importing torchvision or open_clip

"""

import torch
import torch.utils._pytree

# Add the missing function if it doesn't exist
if not hasattr(torch.utils._pytree, 'register_pytree_node'):
    print(f"  Patching torch.utils._pytree (torch version: {torch.__version__})")

    def _dummy_register_pytree_node(*args, **kwargs):
        """No-op placeholder for older torch versions."""
        pass

    torch.utils._pytree.register_pytree_node = _dummy_register_pytree_node
    print("  ✓ Patch applied successfully")
else:
    print(f"  ✓ No patch needed (torch version: {torch.__version__})")

# Verify the patch works by importing torchvision
try:
    import torchvision
    print(f"  ✓ torchvision imported OK (version: {torchvision.__version__})")
except Exception as e:
    print(f"  ✗ torchvision import failed: {e}")

# Check if open_clip works now
try:
    import open_clip
    print(f"  ✓ open_clip imported OK")
    CLIP_AVAILABLE = True
except ImportError:
    print(f"  ✗ open_clip not installed, will use ResNet50 only")
    CLIP_AVAILABLE = False
except Exception as e:
    print(f"  ✗ open_clip import error: {e}")
    CLIP_AVAILABLE = False

print(f"\n  CUDA available: {torch.cuda.is_available()}")

  Patching torch.utils._pytree (torch version: 2.1.2+cu121)
  ✓ Patch applied successfully
  ✓ torchvision imported OK (version: 0.16.2+cu121)


/home/qingqisong/.local/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


  ✓ open_clip imported OK

  CUDA available: True


In [ ]:
# ============================================================================
# EXPERIMENT E: POST-LASSO ANALYSIS (FIXED VERSION)
# ============================================================================

print(f"\n\n{'='*80}")
print("EXPERIMENT E: POST-LASSO ANALYSIS (FIXED)")
print("  → Step 1: Lasso variable selection")
print("  → Step 2: Refit logistic regression on selected variables")
print("  → Step 3: Statistical significance testing")
print(f"{'='*80}")

import statsmodels.api as sm
from scipy.stats import norm

# Helper function for significance stars
def sig_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    return ''

# ============================================================================
# E1: Prepare Full Feature Matrix (Tabular + PCA100 Image)
# ============================================================================

print("\n[E1] Preparing feature matrix...")

# Use PCA 100 for both ResNet and CLIP
X_tr_r100, X_te_r100, var_r, comp_r = create_pca_features(
    resnet_o_raw, resnet_d_raw, 100, train_idx, test_idx
)
X_tr_c100, X_te_c100, var_c, comp_c = create_pca_features(
    clip_o_raw, clip_d_raw, 100, train_idx, test_idx
)

# Combine all features
X_tr_full = np.column_stack([X_tab_train, X_tr_r100, X_tr_c100])
X_te_full = np.column_stack([X_tab_test, X_te_r100, X_te_c100])

# Feature names
img_feat_names = (
    [f'O_resnet_pc{i+1}' for i in range(comp_r)] +
    [f'D_resnet_pc{i+1}' for i in range(comp_r)] +
    [f'O_clip_pc{i+1}' for i in range(comp_c)] +
    [f'D_clip_pc{i+1}' for i in range(comp_c)]
)
all_feat_names = tabular_feats + img_feat_names

print(f"  Total features: {len(all_feat_names)}")
print(f"    Tabular: {len(tabular_feats)}")
print(f"    Image (ResNet PCA): {comp_r * 2}")
print(f"    Image (CLIP PCA): {comp_c * 2}")

# Scale full feature matrix for Lasso
scaler_full = StandardScaler()
X_tr_scaled = scaler_full.fit_transform(X_tr_full)
X_te_scaled = scaler_full.transform(X_te_full)

print(f"  Train samples: {X_tr_scaled.shape[0]}")
print(f"  Test samples: {X_te_scaled.shape[0]}")

# ============================================================================
# E2: LASSO VARIABLE SELECTION WITH DIFFERENT C VALUES
# ============================================================================

print("\n[E2] Lasso variable selection at different regularization strengths...")

C_values_lasso = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]

lasso_results = []

print(f"\n  {'C':>8} {'Non-zero':>10} {'Tab':>6} {'ResNet':>8} {'CLIP':>8} "
      f"{'AUC':>8} {'Acc':>8}")
print(f"  {'-'*65}")

for C in C_values_lasso:
    lasso_model = LogisticRegression(
        penalty='l1', C=C, solver='liblinear', max_iter=3000, random_state=42
    )
    lasso_model.fit(X_tr_scaled, y_train)

    coefs = lasso_model.coef_[0]
    nonzero_mask = coefs != 0
    n_nonzero = nonzero_mask.sum()

    # Count by category
    n_tab = sum(1 for i, name in enumerate(all_feat_names)
                if nonzero_mask[i] and name in tabular_feats)
    n_resnet = sum(1 for i, name in enumerate(all_feat_names)
                   if nonzero_mask[i] and 'resnet' in name)
    n_clip = sum(1 for i, name in enumerate(all_feat_names)
                 if nonzero_mask[i] and 'clip' in name)

    # Performance
    y_prob = lasso_model.predict_proba(X_te_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, lasso_model.predict(X_te_scaled))

    lasso_results.append({
        'C': C,
        'n_nonzero': n_nonzero,
        'n_tab': n_tab,
        'n_resnet': n_resnet,
        'n_clip': n_clip,
        'auc': auc,
        'acc': acc,
        'nonzero_mask': nonzero_mask,
        'coefs': coefs
    })

    print(f"  {C:>8.3f} {n_nonzero:>10} {n_tab:>6} {n_resnet:>8} {n_clip:>8} "
          f"{auc:>8.4f} {acc:>8.4f}")

# ============================================================================
# E3: POST-LASSO SIGNIFICANCE TESTING
# ============================================================================

print(f"\n\n[E3] Post-Lasso Significance Testing")
print("  → Refit standard logistic regression on Lasso-selected variables")
print("  → Compute z-statistics and p-values")

# We'll test multiple C values for robustness
C_values_test = [0.01, 0.02, 0.05]

for C_selected in C_values_test:
    print(f"\n  {'='*70}")
    print(f"  POST-LASSO ANALYSIS WITH C = {C_selected}")
    print(f"  {'='*70}")

    # Get the Lasso result for selected C
    lasso_sel = [r for r in lasso_results if r['C'] == C_selected][0]
    selected_mask = lasso_sel['nonzero_mask']
    selected_indices = np.where(selected_mask)[0]
    selected_names = [all_feat_names[i] for i in selected_indices]

    print(f"\n  Variables selected by Lasso: {len(selected_names)}")

    if len(selected_names) == 0:
        print("  No variables selected, skipping...")
        continue

    # Extract selected features for train and test
    X_tr_selected = X_tr_scaled[:, selected_indices]
    X_te_selected = X_te_scaled[:, selected_indices]

    print(f"  X_train shape: {X_tr_selected.shape}")
    print(f"  y_train shape: {y_train.shape}")

    # Fit statsmodels Logit for significance testing
    # Add constant for intercept
    X_tr_sm = sm.add_constant(X_tr_selected, has_constant='add')
    X_te_sm = sm.add_constant(X_te_selected, has_constant='add')

    print(f"  X_train with constant shape: {X_tr_sm.shape}")

    print("\n  Fitting statsmodels Logit model...")

    try:
        logit_model = sm.Logit(y_train, X_tr_sm)
        logit_result = logit_model.fit(disp=0, maxiter=1000, method='lbfgs')

        # Model fit statistics
        ll_model = logit_result.llf
        ll_null = logit_result.llnull
        pseudo_r2 = logit_result.prsquared
        aic = logit_result.aic
        bic = logit_result.bic

        print(f"\n  ┌─────────────────────────────────────────────────────────────┐")
        print(f"  │  POST-LASSO MODEL FIT STATISTICS (C={C_selected})                     │")
        print(f"  ├─────────────────────────────────────────────────────────────┤")
        print(f"  │  Log-Likelihood (model):    {ll_model:>15.2f}               │")
        print(f"  │  Log-Likelihood (null):     {ll_null:>15.2f}               │")
        print(f"  │  Pseudo R² (McFadden):      {pseudo_r2:>15.4f}               │")
        print(f"  │  AIC:                       {aic:>15.2f}               │")
        print(f"  │  BIC:                       {bic:>15.2f}               │")
        print(f"  │  N observations:            {len(y_train):>15,}               │")
        print(f"  │  N parameters:              {len(selected_names) + 1:>15}               │")
        print(f"  └─────────────────────────────────────────────────────────────┘")

        # Test set performance
        y_prob_postlasso = logit_result.predict(X_te_sm)
        auc_postlasso = roc_auc_score(y_test, y_prob_postlasso)
        acc_postlasso = accuracy_score(y_test, (y_prob_postlasso > 0.5).astype(int))

        print(f"\n  Test Set Performance:")
        print(f"    AUC:      {auc_postlasso:.4f}")
        print(f"    Accuracy: {acc_postlasso:.4f}")

        # Extract coefficients, std errors, z-values, p-values
        # Note: params[0] is intercept, params[1:] are feature coefficients
        coef_data = []

        # Intercept
        coef_data.append({
            'variable': 'intercept',
            'coef': logit_result.params[0],
            'std_err': logit_result.bse[0],
            'z': logit_result.tvalues[0],
            'p_value': logit_result.pvalues[0],
            'category': 'Intercept'
        })

        # Features
        for i, name in enumerate(selected_names):
            idx = i + 1  # +1 for intercept

            # Determine category
            if name in person_hh:
                cat = 'Person/HH'
            elif name in trip_f:
                cat = 'Trip'
            elif name in orig_be or name in dest_be:
                cat = 'Zone BE'
            elif 'resnet' in name:
                cat = 'ResNet'
            elif 'clip' in name:
                cat = 'CLIP'
            else:
                cat = 'Other'

            coef_data.append({
                'variable': name,
                'coef': logit_result.params[idx],
                'std_err': logit_result.bse[idx],
                'z': logit_result.tvalues[idx],
                'p_value': logit_result.pvalues[idx],
                'category': cat
            })

        coef_df = pd.DataFrame(coef_data)
        coef_df['abs_coef'] = np.abs(coef_df['coef'])
        coef_df['sig'] = coef_df['p_value'].apply(sig_stars)
        coef_df_sorted = coef_df.sort_values('abs_coef', ascending=False)

        # Print top coefficients
        print(f"\n  TOP 30 COEFFICIENTS (sorted by |coef|):")
        print(f"  {'Variable':<30} {'Cat':<10} {'Coef':>10} {'SE':>10} {'z':>8} {'p':>10} {'Sig':>5}")
        print(f"  {'-'*90}")

        for _, row in coef_df_sorted.head(30).iterrows():
            print(f"  {row['variable']:<30} {row['category']:<10} {row['coef']:>10.4f} "
                  f"{row['std_err']:>10.4f} {row['z']:>8.3f} {row['p_value']:>10.4f} {row['sig']:>5}")

        # Summary by category
        print(f"\n  SIGNIFICANCE SUMMARY BY CATEGORY:")
        print(f"  {'Category':<15} {'Total':>8} {'p<0.001':>10} {'p<0.01':>10} {'p<0.05':>10} {'p<0.1':>10}")
        print(f"  {'-'*65}")

        for cat in ['Person/HH', 'Trip', 'Zone BE', 'ResNet', 'CLIP']:
            cat_df = coef_df[coef_df['category'] == cat]
            n_total = len(cat_df)
            if n_total > 0:
                n_001 = (cat_df['p_value'] < 0.001).sum()
                n_01 = (cat_df['p_value'] < 0.01).sum()
                n_05 = (cat_df['p_value'] < 0.05).sum()
                n_1 = (cat_df['p_value'] < 0.1).sum()
                print(f"  {cat:<15} {n_total:>8} {n_001:>10} {n_01:>10} {n_05:>10} {n_1:>10}")

        # Detailed image features
        img_df = coef_df[coef_df['category'].isin(['ResNet', 'CLIP'])]
        sig_img = img_df[img_df['p_value'] < 0.05].sort_values('abs_coef', ascending=False)

        print(f"\n  SIGNIFICANT IMAGE FEATURES (p < 0.05): {len(sig_img)}")
        if len(sig_img) > 0:
            print(f"  {'Variable':<25} {'Coef':>10} {'z':>8} {'p-value':>12} {'Direction':<12}")
            print(f"  {'-'*70}")
            for _, row in sig_img.head(20).iterrows():
                direction = "↑ transit" if row['coef'] > 0 else "↓ transit"
                print(f"  {row['variable']:<25} {row['coef']:>10.4f} {row['z']:>8.3f} "
                      f"{row['p_value']:>12.6f} {direction:<12}")

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback
        traceback.print_exc()


# ============================================================================
# E4: FOCUSED ANALYSIS WITH C=0.02 (Good balance of sparsity and signal)
# ============================================================================

print(f"\n\n{'='*80}")
print("[E4] DETAILED POST-LASSO ANALYSIS WITH C=0.02")
print(f"{'='*80}")

C_focus = 0.02
lasso_sel = [r for r in lasso_results if r['C'] == C_focus][0]
selected_mask = lasso_sel['nonzero_mask']
selected_indices = np.where(selected_mask)[0]
selected_names = [all_feat_names[i] for i in selected_indices]

print(f"\n  Selected {len(selected_names)} variables")

# Separate by category
tab_selected = [n for n in selected_names if n in tabular_feats]
be_selected = [n for n in selected_names if n in orig_be or n in dest_be]
resnet_selected = [n for n in selected_names if 'resnet' in n]
clip_selected = [n for n in selected_names if 'clip' in n]

print(f"    Person/HH + Trip: {len([n for n in tab_selected if n in person_hh or n in trip_f])}")
print(f"    Zone BE:          {len(be_selected)}")
print(f"    ResNet:           {len(resnet_selected)}")
print(f"    CLIP:             {len(clip_selected)}")

# Fit Post-Lasso model
X_tr_sel = X_tr_scaled[:, selected_indices]
X_te_sel = X_te_scaled[:, selected_indices]
X_tr_sm = sm.add_constant(X_tr_sel, has_constant='add')
X_te_sm = sm.add_constant(X_te_sel, has_constant='add')

logit_model = sm.Logit(y_train, X_tr_sm)
result = logit_model.fit(disp=0, maxiter=1000, method='lbfgs')

# Build coefficient dataframe
coef_list = []
for i, name in enumerate(selected_names):
    idx = i + 1
    if name in person_hh:
        cat = 'Person/HH'
    elif name in trip_f:
        cat = 'Trip'
    elif name in orig_be or name in dest_be:
        cat = 'Zone BE'
    elif 'resnet' in name:
        cat = 'ResNet'
    elif 'clip' in name:
        cat = 'CLIP'
    else:
        cat = 'Other'

    coef_list.append({
        'variable': name,
        'coef': result.params[idx],
        'std_err': result.bse[idx],
        'z': result.tvalues[idx],
        'p_value': result.pvalues[idx],
        'category': cat,
        'sig': sig_stars(result.pvalues[idx])
    })

coef_df = pd.DataFrame(coef_list)
coef_df['abs_coef'] = np.abs(coef_df['coef'])

# Print all significant image features
print(f"\n  ALL SIGNIFICANT IMAGE FEATURES (p < 0.05):")
sig_img = coef_df[(coef_df['category'].isin(['ResNet', 'CLIP'])) & (coef_df['p_value'] < 0.05)]
sig_img = sig_img.sort_values('abs_coef', ascending=False)

print(f"\n  Found {len(sig_img)} significant image features")
print(f"\n  {'#':>3} {'Variable':<25} {'Category':<8} {'Coef':>10} {'z':>8} {'p-value':>12}")
print(f"  {'-'*75}")

for rank, (_, row) in enumerate(sig_img.iterrows(), 1):
    print(f"  {rank:>3} {row['variable']:<25} {row['category']:<8} {row['coef']:>10.4f} "
          f"{row['z']:>8.3f} {row['p_value']:>12.6f}")

# Breakdown by Origin vs Destination
print(f"\n  BREAKDOWN BY LOCATION:")
sig_origin = sig_img[sig_img['variable'].str.startswith('O_')]
sig_dest = sig_img[sig_img['variable'].str.startswith('D_')]
print(f"    Origin features (significant): {len(sig_origin)}")
print(f"    Destination features (significant): {len(sig_dest)}")

# Breakdown by ResNet vs CLIP
sig_resnet = sig_img[sig_img['category'] == 'ResNet']
sig_clip = sig_img[sig_img['category'] == 'CLIP']
print(f"    ResNet features (significant): {len(sig_resnet)}")
print(f"    CLIP features (significant): {len(sig_clip)}")


# ============================================================================
# E5: COMPARE TABULAR-ONLY VS POST-LASSO MODEL
# ============================================================================

print(f"\n\n{'='*80}")
print("[E5] MODEL COMPARISON: Tabular-Only vs Post-Lasso")
print(f"{'='*80}")

# Model 1: Tabular only
X_tab_sm_train = sm.add_constant(X_tab_train, has_constant='add')
X_tab_sm_test = sm.add_constant(X_tab_test, has_constant='add')

logit_tab = sm.Logit(y_train, X_tab_sm_train)
result_tab = logit_tab.fit(disp=0, maxiter=1000, method='lbfgs')

y_prob_tab = result_tab.predict(X_tab_sm_test)
auc_tab = roc_auc_score(y_test, y_prob_tab)

# Model 2: Post-Lasso with C=0.02
y_prob_postlasso = result.predict(X_te_sm)
auc_postlasso = roc_auc_score(y_test, y_prob_postlasso)

print(f"\n  ┌─────────────────────────────────────────────────────────────────────────┐")
print(f"  │  MODEL COMPARISON                                                       │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  {'Metric':<30} {'Tabular Only':>16} {'Post-Lasso':>18}   │")
print(f"  │  {'-'*68}│")
print(f"  │  {'N parameters':<30} {len(tabular_feats)+1:>16} {len(selected_names)+1:>18}   │")
print(f"  │  {'Log-likelihood':<30} {result_tab.llf:>16.2f} {result.llf:>18.2f}   │")
print(f"  │  {'Pseudo R² (McFadden)':<30} {result_tab.prsquared:>16.4f} {result.prsquared:>18.4f}   │")
print(f"  │  {'AIC':<30} {result_tab.aic:>16.1f} {result.aic:>18.1f}   │")
print(f"  │  {'BIC':<30} {result_tab.bic:>16.1f} {result.bic:>18.1f}   │")
print(f"  │  {'Test AUC':<30} {auc_tab:>16.4f} {auc_postlasso:>18.4f}   │")
print(f"  └─────────────────────────────────────────────────────────────────────────┘")

# Change in metrics
print(f"\n  IMPROVEMENT FROM ADDING IMAGE FEATURES:")
print(f"    Δ Pseudo R²:  {result.prsquared - result_tab.prsquared:>+.4f}")
print(f"    Δ AUC:        {auc_postlasso - auc_tab:>+.4f}")
print(f"    Δ LL:         {result.llf - result_tab.llf:>+.2f}")

# Likelihood ratio test (approximate - models not strictly nested)
lr_stat = 2 * (result.llf - result_tab.llf)
df_diff = len(selected_names) - len(tabular_feats)
if lr_stat > 0 and df_diff > 0:
    p_lr = 1 - stats.chi2.cdf(lr_stat, df_diff)
    print(f"\n  Likelihood Ratio Test (approximate):")
    print(f"    LR statistic: {lr_stat:.2f}")
    print(f"    df:           {df_diff}")
    print(f"    p-value:      {p_lr:.2e}")
    if p_lr < 0.001:
        print(f"    → Image features contribute significantly (p < 0.001)")


# ============================================================================
# E6: ROBUSTNESS CHECK - VARY C AND CHECK CONSISTENCY OF SIGNIFICANT FEATURES
# ============================================================================

print(f"\n\n{'='*80}")
print("[E6] ROBUSTNESS CHECK: Consistency of Significant Image Features")
print(f"{'='*80}")

# Track which image features are significant across different C values
C_robustness = [0.01, 0.02, 0.05, 0.1]
img_sig_counts = {}

for C in C_robustness:
    lasso_sel = [r for r in lasso_results if r['C'] == C][0]
    selected_mask = lasso_sel['nonzero_mask']
    selected_indices = np.where(selected_mask)[0]
    selected_names_c = [all_feat_names[i] for i in selected_indices]

    if len(selected_names_c) == 0:
        continue

    X_tr_sel = X_tr_scaled[:, selected_indices]
    X_tr_sm = sm.add_constant(X_tr_sel, has_constant='add')

    try:
        logit_model = sm.Logit(y_train, X_tr_sm)
        result_c = logit_model.fit(disp=0, maxiter=1000, method='lbfgs')

        for i, name in enumerate(selected_names_c):
            if 'resnet' in name or 'clip' in name:
                p_val = result_c.pvalues[i + 1]
                if p_val < 0.05:
                    if name not in img_sig_counts:
                        img_sig_counts[name] = []
                    img_sig_counts[name].append(C)
    except:
        continue

# Find consistently significant features
print(f"\n  Image features significant (p<0.05) across multiple C values:")
print(f"\n  {'Variable':<30} {'C values where significant'}")
print(f"  {'-'*60}")

consistent_features = []
for name, c_list in sorted(img_sig_counts.items(), key=lambda x: -len(x[1])):
    if len(c_list) >= 2:
        consistent_features.append(name)
        c_str = ', '.join([f'{c:.2f}' for c in c_list])
        print(f"  {name:<30} {c_str}")

print(f"\n  Features significant in ≥2 C values: {len(consistent_features)}")
print(f"  Features significant in all 4 C values: {sum(1 for n, c in img_sig_counts.items() if len(c) == 4)}")


# ============================================================================
# E7: INTERPRETATION OF TOP IMAGE FEATURES
# ============================================================================

print(f"\n\n{'='*80}")
print("[E7] INTERPRETATION OF SIGNIFICANT IMAGE FEATURES")
print(f"{'='*80}")

# Use C=0.02 results
print(f"\n  Using Post-Lasso results with C=0.02")
print(f"\n  INTERPRETATION GUIDE:")
print(f"  • Positive coefficient → higher values associated with MORE transit use")
print(f"  • Negative coefficient → higher values associated with LESS transit use")
print(f"  • PCA components capture combinations of visual patterns")
print(f"  • Lower PC numbers capture more variance")

# Top positive coefficients (associated with transit use)
sig_img = coef_df[(coef_df['category'].isin(['ResNet', 'CLIP'])) & (coef_df['p_value'] < 0.05)]

pos_img = sig_img[sig_img['coef'] > 0].sort_values('coef', ascending=False)
neg_img = sig_img[sig_img['coef'] < 0].sort_values('coef', ascending=True)

print(f"\n  IMAGE FEATURES POSITIVELY ASSOCIATED WITH TRANSIT USE:")
print(f"  (Higher values → more likely to use transit)")
print(f"  {'Variable':<25} {'Coef':>10} {'z':>8} {'p-value':>12}")
print(f"  {'-'*60}")
for _, row in pos_img.head(10).iterrows():
    print(f"  {row['variable']:<25} {row['coef']:>10.4f} {row['z']:>8.3f} {row['p_value']:>12.6f}")

print(f"\n  IMAGE FEATURES NEGATIVELY ASSOCIATED WITH TRANSIT USE:")
print(f"  (Higher values → less likely to use transit)")
print(f"  {'Variable':<25} {'Coef':>10} {'z':>8} {'p-value':>12}")
print(f"  {'-'*60}")
for _, row in neg_img.head(10).iterrows():
    print(f"  {row['variable']:<25} {row['coef']:>10.4f} {row['z']:>8.3f} {row['p_value']:>12.6f}")


# ============================================================================
# FINAL SUMMARY
# ============================================================================

print(f"\n\n{'='*80}")
print("FINAL SUMMARY: POST-LASSO ANALYSIS")
print(f"{'='*80}")

print(f"""
┌──────────────────────────────────────────────────────────────────────────────┐
│  POST-LASSO KEY FINDINGS                                                     │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  1. VARIABLE SELECTION (Lasso C=0.02):                                       │
│     • Total variables selected: {len(selected_names):>4}                                        │
│     • Tabular features:         {len(tab_selected) + len(be_selected):>4}                                        │
│     • Image features:           {len(resnet_selected) + len(clip_selected):>4}                                        │
│                                                                              │
│  2. STATISTICAL SIGNIFICANCE (Post-Lasso p < 0.05):                          │
│     • Total significant image features: {len(sig_img):>4}                               │
│     • ResNet features:                  {len(sig_resnet):>4}                               │
│     • CLIP features:                    {len(sig_clip):>4}                               │
│                                                                              │
│  3. MODEL COMPARISON:                                                        │
│     • Tabular-only AUC:    {auc_tab:.4f}                                       │
│     • Post-Lasso AUC:      {auc_postlasso:.4f} (Δ = {auc_postlasso - auc_tab:+.4f})                          │
│     • Pseudo R² improved:  {result_tab.prsquared:.4f} → {result.prsquared:.4f} (Δ = {result.prsquared - result_tab.prsquared:+.4f})         │
│                                                                              │
│  4. ROBUSTNESS:                                                              │
│     • Features significant in ≥2 C values: {len(consistent_features):>4}                          │
│                                                                              │
│  CONCLUSION:                                                                 │
│  ───────────                                                                 │
│  Image embeddings contain STATISTICALLY SIGNIFICANT information              │
│  about mode choice, even after controlling for tabular features.             │
│  However, the PRACTICAL improvement is modest (Δ AUC ≈ {auc_postlasso - auc_tab:+.3f}).              │
│                                                                              │
│  This suggests that satellite imagery captures urban patterns that           │
│  are RELATED TO but NOT FULLY CAPTURED BY traditional BE metrics.            │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
""")

print("=" * 80)
print("  EXPERIMENT E (POST-LASSO) COMPLETE")
print("=" * 80)



EXPERIMENT E: POST-LASSO ANALYSIS (FIXED)
  → Step 1: Lasso variable selection
  → Step 2: Refit logistic regression on selected variables
  → Step 3: Statistical significance testing

[E1] Preparing feature matrix...
  Total features: 437
    Tabular: 37
    Image (ResNet PCA): 200
    Image (CLIP PCA): 200
  Train samples: 10410
  Test samples: 2603

[E2] Lasso variable selection at different regularization strengths...

         C   Non-zero    Tab   ResNet     CLIP      AUC      Acc
  -----------------------------------------------------------------
     0.001          2      2        0        0   0.8398   0.8210
     0.005         14     11        2        1   0.8566   0.8333
     0.010         21     13        4        4   0.8634   0.8417
     0.020         87     20       28       39   0.8639   0.8467
     0.050        204     25       82       97   0.8560   0.8452
     0.100        303     30      130      143   0.8487   0.8433
     0.200        352     31      153      168  